# traffic-cv: обучение на GPU (Kaggle / Colab)

Секреты: `DAGSHUB_USER`, `DAGSHUB_TOKEN`.
- Kaggle: Add-ons → Secrets, включить оба для ноутбука. Settings → Accelerator: GPU, Internet: on.
- Colab: значок ключа слева, включить доступ ноутбука. Среда выполнения → GPU T4.

На ночь: Save Version → Save & Run All (Commit). Лимит 12 ч. Каждая модель сразу уходит в DVC-хранилище, метрики по эпохам видны на DagsHub → Experiments.

In [ ]:
!nvidia-smi
!git clone https://github.com/jestersw/trafic-cv.git
%cd trafic-cv
!pip install -q -e ".[train,track,eda,data]"

In [ ]:
import os

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    from kaggle_secrets import UserSecretsClient

    get_secret = UserSecretsClient().get_secret
    PLATFORM = "kaggle"
else:
    from google.colab import userdata

    get_secret = userdata.get
    PLATFORM = "colab"

user = get_secret("DAGSHUB_USER")
token = get_secret("DAGSHUB_TOKEN")
repo = f"https://dagshub.com/{user}/trafic-cv"

os.environ["MLFLOW_TRACKING_URI"] = f"{repo}.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = user
os.environ["MLFLOW_TRACKING_PASSWORD"] = token

!dvc remote modify dagshub --local access_key_id {token}
!dvc remote modify dagshub --local secret_access_key {token}
print(PLATFORM, repo)

## Данные

In [ ]:
!dvc pull -j 16 data/processed/detrac
!ls data/processed/detrac/images/train | wc -l

## Детекция (этап 2)

Трекинг (этап 3) считается локально: ему нужны все кадры test-видео из `data/raw`, их нет в DVC-хранилище.

In [ ]:
!dvc pull models/baseline.pt
!cat reports/detection/baseline.json

In [ ]:
!dvc repro -s train_finetune
!dvc push -j 16 models/finetune.pt
!cat reports/detection/finetune.json

In [ ]:
!dvc repro -s compare_detection

In [ ]:
!cat reports/detection/compare.md

## Сохранить результаты

Модели уходят в DVC-хранилище. `results.zip` (`dvc.lock` + отчёты) нужно положить в локальный репозиторий и закоммитить.
- Kaggle: вкладка Output у версии ноутбука.
- Colab: скачается автоматически.

In [ ]:
!dvc push -j 16
!zip -qr results.zip dvc.lock reports
if PLATFORM == "colab":
    from google.colab import files

    files.download("results.zip")
else:
    !cp results.zip /kaggle/working/
    print("results.zip -> Output")